Checking for the residual tuning after removing the activity predict by shelter distance.
Not quite TunED!

In [ ]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

# JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# "JAL3_7sept", "JAL3_4sept", "JAL3_1sept", "JAL3_25aug", "JAL3_22aug",

experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip3_18mar, JAL6_flip5_25mar, # (unmatched number of neurons and cluster ids) # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_14may, JAL8_flip4_10may]

In [ ]:
%load_ext autoreload
from behave_analysis.utils.creating_directories import make_directory
from JR_test_scripts.escape.escape_utils import load, load_homing
from JR_test_scripts.escape.escape_tuning_funcs import tuning_method
from JR_test_scripts.escape.escape_data_loading_funcs import extract_homing_and_escape_periods, build_shift_vector, compute_dist_shelt

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

%matplotlib inline

In [ ]:
"""Check tuning of residual using linear shift stats"""
%autoreload 2
Nbins = 50

for exp in experiments_objects:

    # 1. load in explore tuning curve
    nickname = exp.nick_name + '_' + exp.experiment_date
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'bird_dist_shelter'
    dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves_explore/")
    saving_file = dump_path + exp_nickname + '_Tuning.npz'
    data = np.load(saving_file)

    # pull out the params
    data_files = data.files
    exp_fr_real = data['fr_real']

    # 2. load in homing/escape tuning curve
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'escape'
    dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
    saving_file = dump_path + exp_nickname + '_ProperTuning.npz'
    data = np.load(saving_file)

    data_files = data.files
    fr_real = data["fr_real"]

    # 3. load in neural data
    session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, escape, outofshelter = load(exp)
    ons, offs, homie = load_homing(session, len(behave))
    fcm = gaussian_filter1d(frame_by_cluster_matrix, 2, axis = 0)

    # 4. set up shifts
    # setting up the shifts
    min_step = 120
    step = 400
    step_n = 100
    shifts_one_sided = np.arange(min_step,min_step+((step_n/2)*step), step)

    shifts, shift_vector = build_shift_vector(bar, barflip, session, ons, offs, shifts_one_sided)

    # 4. extract 'real stat' escape matrix with respect to 'escape'
    comp = 'escape'
    if comp == 'escape':
        bins = np.arange(0,1,1/Nbins)
    frac_escape, escape_matrix, cond, h_start = extract_homing_and_escape_periods(session, 
                                                                                    fcm[shift_vector,:], 
                                                                                    behave[shift_vector], 
                                                                                    y_pos[shift_vector], 
                                                                                    x_pos[shift_vector], 
                                                                                    bar[shift_vector], # should this also be shifted?!?! 
                                                                                    barflip[shift_vector], 
                                                                                    comp, 
                                                                                    ons, offs, shift_vector, 
                                                                                    bins = bins,
                                                                                    no_stationary = False, 
                                                                                    return_escape = False,
                                                                                    zscore = False)

    # initialize vars
    n_neur = frame_by_cluster_matrix.shape[1]
    n_cond = len(np.unique(cond))

    # 5. extract behavioral data with respect to 'distance to shelter'
    complete_shelter_dist = compute_dist_shelt(x_pos, y_pos, cond=np.zeros_like(x_pos), session=session)
    shelt_dist, _, _, _ = extract_homing_and_escape_periods(session, 
                                                            fcm[shift_vector,:], 
                                                            behave[shift_vector], 
                                                            y_pos[shift_vector], 
                                                            x_pos[shift_vector], 
                                                            bar[shift_vector], # should this also be shifted?!?! 
                                                            barflip[shift_vector], 
                                                            'bird_dist_shelter', 
                                                            ons, offs, shift_vector, 
                                                            bins = np.arange(0,np.amax(complete_shelter_dist),np.amax(complete_shelter_dist)/Nbins),
                                                            no_stationary = False, 
                                                            return_escape = False,
                                                            zscore = False)

    # 6. create predicted neural activity using tuning to distance to shelter in exploration
    # I think that if exp_fr_full.shape[2] is greater than bins_dist, those latter values can be ignored because the mouse can be further in exploration, but the lower values should match

    exp_predicted_matrix = np.full_like(escape_matrix, np.nan)

    for n in range(n_neur):
        for c in range(n_cond):
            u = shelt_dist[cond == c].astype(int)
            v = exp_fr_real[c, n, :]
            exp_predicted_matrix[n, cond == c] = v[u]

    # 7. subtract predicted neural activity from actual neural activity
    exp_residual_matrix = escape_matrix - exp_predicted_matrix

    # 8. compute tuning curve for the residuals
    y_fitted_real_exp, R_real_exp, fr_real_exp, params_real_exp, mat_real_cond_exp = tuning_method(frac_escape, 
                                                                                                    exp_residual_matrix, 
                                                                                                    cond, 
                                                                                                    h_start, 
                                                                                                    Nbins, 
                                                                                                    n_cond,
                                                                                                    n_neur)

    # 10. create predicted neural activity using tuning to fraction in escape
    # This gives us a sense of how good we can expect to predict the full trace, there should be no tuning here

    predicted_matrix = np.full_like(escape_matrix, np.nan)

    for n in range(n_neur):
        for c in range(n_cond):
            u = frac_escape[cond == c].astype(int)
            v = fr_real[c, n, :]
            predicted_matrix[n, cond == c] = v[u]


    # 11. subtract predicted neural activity from actual neural activity
    residual_matrix = escape_matrix - predicted_matrix

    # 12. compute tuning curve for the residuals
    y_fitted_real_res, R_real_res, fr_real_res, params_real_res, mat_real_cond_res = tuning_method(frac_escape, 
                                                                                                    residual_matrix, 
                                                                                                    cond, 
                                                                                                    h_start, 
                                                                                                    Nbins, 
                                                                                                    n_cond,
                                                                                                    n_neur)

    # 13. perform linear shifts
    y_fitted_shift_res = np.full((step_n, n_cond, n_neur, Nbins), np.nan) # conditions x neurons x n_bins
    R_shift_res = np.zeros((step_n,n_neur, n_cond)) # neurons x conditions
    params_shifts_res = np.zeros((step_n,n_neur, n_cond, 6)) # neurons x conditions
    fr_shift_res = np.full((step_n, n_cond, n_neur, Nbins), np.nan)
    c = [len([x for x in h_start if cond[x] == i]) for i in range(3)] # trial n per condition
    mat_shift_cond_res = np.full((step_n, n_cond, n_neur, max(c), Nbins), np.nan)

    y_fitted_shift_exp_res = np.full((step_n, n_cond, n_neur, Nbins), np.nan) # conditions x neurons x n_bins
    R_shift_exp_res = np.zeros((step_n,n_neur, n_cond)) # neurons x conditions
    params_shifts_exp_res = np.zeros((step_n,n_neur, n_cond, 6)) # neurons x conditions
    fr_shift_exp_res = np.full((step_n, n_cond, n_neur, Nbins), np.nan)
    c = [len([x for x in h_start if cond[x] == i]) for i in range(3)] # trial n per condition
    mat_shift_cond_exp_res = np.full((step_n, n_cond, n_neur, max(c), Nbins), np.nan)

    for s_idx, s in enumerate(shifts):
        shifted_vec = np.roll(shift_vector,int(s))

        _, esc_mat_esc_shift, _, _ = extract_homing_and_escape_periods(session, 
                                                                        fcm[shifted_vec,:], 
                                                                        behave[shift_vector], 
                                                                        y_pos[shift_vector], 
                                                                        x_pos[shift_vector], 
                                                                        bar[shift_vector], # should this also be shifted?!?! 
                                                                        barflip[shift_vector], 
                                                                        comp, 
                                                                        ons, offs, shift_vector, 
                                                                        bins = bins,
                                                                        no_stationary = False, 
                                                                        return_escape = False,
                                                                        zscore = False)

        # 7. subtract predicted neural activity from actual neural activity
        exp_residual_matrix_shift = esc_mat_esc_shift - exp_predicted_matrix

        # 8. compute tuning curve for the residuals
        y_fitted_shift_exp_res[s_idx,:,:,:], R_shift_exp_res[s_idx,:,:], fr_shift_exp_res[s_idx,:,:,:], params_shifts_exp_res[s_idx,:,:,:], mat_shift_cond_exp_res[s_idx,:,:,:] = tuning_method(frac_escape, 
                                                                                                                                                                                                exp_residual_matrix_shift, 
                                                                                                                                                                                                cond, 
                                                                                                                                                                                                h_start, 
                                                                                                                                                                                                Nbins, 
                                                                                                                                                                                                n_cond,
                                                                                                                                                                                                n_neur)

        # 11. subtract predicted neural activity from actual neural activity
        residual_matrix_shift = esc_mat_esc_shift - predicted_matrix

        # 12. compute tuning curve for the residuals
        y_fitted_shift_res[s_idx,:,:,:], R_shift_res[s_idx,:,:], fr_shift_res[s_idx,:,:,:], params_shifts_res[s_idx,:,:,:], mat_shift_cond_res[s_idx,:,:,:] = tuning_method(frac_escape, 
                                                                                                                                                                            residual_matrix_shift, 
                                                                                                                                                                            cond, 
                                                                                                                                                                            h_start, 
                                                                                                                                                                            Nbins, 
                                                                                                                                                                            n_cond,
                                                                                                                                                                            n_neur)

    """Save data"""
    dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
    saving_file = dump_path + exp_nickname + '_ResidualsTuning'
    np.savez(saving_file, 
            # data from shifted residuals of tuning to % escape
            R_shift_res=R_shift_res, params_shifts_res=params_shifts_res, y_fitted_shift_res=y_fitted_shift_res, 
            fr_shift_res=fr_shift_res, mat_shift_cond_res=mat_shift_cond_res,
            # data from shifted residuals of tuning to distance to shelter
            R_shift_exp_res=R_shift_exp_res, params_shifts_exp_res=params_shifts_exp_res, y_fitted_shift_exp_res=y_fitted_shift_exp_res, 
            fr_shift_exp_res=fr_shift_exp_res, mat_shift_cond_exp_res=mat_shift_cond_exp_res,
            # data from residuals of tuning to % escape
            R_real_res=R_real_res, params_real_res=params_real_res, y_fitted_real_res=y_fitted_real_res, 
            fr_real_res=fr_real_res, mat_real_cond_res=mat_real_cond_res,
            # data from residuals of tuning to distance to shelter
            y_fitted_real_exp=y_fitted_real_exp, R_real_exp=R_real_exp, fr_real_exp=fr_real_exp, params_real_exp=params_real_exp, mat_real_cond_exp=mat_real_cond_exp)

Next cells allow you to 
1. plot some cells and check if the tuning and residulas look right
2. look at the fraction of cells tuned across conditions

In [ ]:
"""Load a session you want to plot"""

exp = experiments_objects[7]

# 1. load in explore tuning curve
exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'bird_dist_shelter'
dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves_explore/")
saving_file = dump_path + exp_nickname + '_Tuning.npz'
data = np.load(saving_file)

# pull out the params
fr_full_exp = data['fr_full']
exp_params_real = data['params_real']
exp_params_shifts = data['params_shifts']

# 2. load in homing/escape tuning curve
exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'escape'
dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
saving_file = dump_path + exp_nickname + '_ProperTuning.npz'
data = np.load(saving_file)

fr_full = data['fr_full']
params_real = data['params_real']
params_shifts = data['params_shifts']

# load in residuals data
dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
saving_file = dump_path + exp_nickname + '_ResidualsTuning.npz'
data = np.load(saving_file)
fr_full_res = data['fr_full']
params_real_exp = data['params_real_exp']
params_shifts_exp_res = data['params_shifts_exp_res']

# identify cells that are sig tuned to %escape in homing/escape
sig_escape = params_real[:,:,0] > np.nanpercentile(params_shifts[:,:,:,0], 95, axis = 0)

# identify cells that are sig tuned to distance to shelter in exploration
exp_sig_dist = exp_params_real[:,:,0] > np.nanpercentile(exp_params_shifts[:,:,:,0], 95, axis = 0)

# ifnd cells whose residual tuning to %escape - distance to shelter in exploration is significant
sig_res = params_real_exp[:,:,0] > np.nanpercentile(params_shifts_exp_res[:,:,:,0], 95, axis = 0)

In [ ]:
"""Plot the tuning to % escape from the real data and the predictions made using the tuning to % escape and dist to shelter in exploration"""
idx = 0
cond = 1
goodones = np.where(np.logical_and(exp_sig_dist[:,cond], sig_escape[:,cond]))[0]

# is the tuning of the residual from %escape significant?
sig = params_real_res[goodones[idx],cond,0] > np.nanpercentile(params_shifts_res[:,goodones[idx],cond,0], 95, axis = 0)
label_escape = r'Residual from %escape tuning'
if sig:
    label_escape = label_escape + ' is sig.'

# is the tuning of the residual from shelt distance in exploration significant?
sig = params_real_exp[goodones[idx],cond,0] > np.nanpercentile(params_shifts_exp_res[:,goodones[idx],cond,0], 95, axis = 0)
label_shelter= 'Residual from shelter dist. tuning'
if sig:
    label_shelter = label_shelter + ' is sig.'

plt.plot(fr_full[cond,goodones[idx],:], label = 'Real')
plt.plot(fr_full_res[cond,goodones[idx],:], label = label_escape)
plt.plot(fr_full_exp[cond,goodones[idx],:], label = label_shelter)
plt.ylabel('Firing Rate (Hz)')
plt.xlabel('Fraction of escape')
plt.legend(loc = 'upper right')
plt.title('Example cell during escape/homing periods')

In [ ]:
"""Bar chart"""

c_names = ['shelter_only', 'barrier', 'flipped_barrier']

fig, axs = plt.subplots(1,3,figsize=(9, 3))
for c in range(3):
    # Compute Fractions
    frac_v1 = np.sum(sig_escape[:,c]) / n_neur
    frac_v2 = np.sum(exp_sig_dist[:,c]) / n_neur
    frac_v1_and_v2 = np.sum(sig_escape[:,c] & exp_sig_dist[:,c]) / n_neur
    frac_all_three = np.sum(sig_escape[:,c] & exp_sig_dist[:,c] & sig_res[:,c]) / n_neur

    # Bar Chart
    categories = ['V1', 'V2', 'V1 & V2', 'All Three']
    fractions = [frac_v1, frac_v2, frac_v1_and_v2, frac_all_three]

    axs[c].bar(categories, fractions, color=['blue', 'green', 'purple', 'orange'])
    axs[c].set_ylabel('Fraction of Total Cells')
    axs[c].set_title('Fraction of Significant Cells \n in ' + c_names[c])
    axs[c].set_ylim(0, 1)  # Since fractions range from 0 to 1
plt.tight_layout()

"""Venn diagram"""
from matplotlib_venn import venn3

fig, axs = plt.subplots(1,3,figsize=(9, 3))
for c in range(3):
    # Compute Set Sizes
    A = np.sum(sig_escape[:,c])  # V1 only
    B = np.sum(exp_sig_dist[:,c])  # V2 only
    C = np.sum(sig_res[:,c])  # V1 regressed
    AB = np.sum(sig_escape[:,c] & exp_sig_dist[:,c])  # Both V1 and V2
    AC = np.sum(sig_escape[:,c] & sig_res[:,c])  # Both V1 and V1 regressed
    BC = np.sum(exp_sig_dist[:,c] & sig_res[:,c])  # Both V2 and V1 regressed
    ABC = np.sum(sig_escape[:,c] & exp_sig_dist[:,c] & sig_res[:,c])  # All three

    # Create Venn Diagram
    venn3(subsets=(A, B, AB, C, AC, BC, ABC), 
        set_labels=('V1', 'V2', 'V1 regressed'), 
        set_colors=('blue', 'red', 'green'),
        alpha=0.5,
        ax=axs[c])
    axs[c].set_title('Overlap of Significant Cells \nin ' + c_names[c])
plt.tight_layout()